<div style="background:#f8fafc; border:1px solid #dbeafe; border-left:7px solid #2563eb; padding:26px 30px; border-radius:16px; font-family:Segoe UI, Arial, sans-serif; color:#1e293b;">

<h1 style="margin-top:0; color:#1d4ed8; font-size:34px;">
⏱️ 04 Early-Warning Dataset - First 50% of Course
</h1>

<p style="font-size:17px; line-height:1.8; color:#334155;">
This notebook builds the <b>second-stage early-warning dataset</b> using only the information available during the first <b>50% of the course timeline</b>.
</p>

<p style="font-size:17px; line-height:1.8; color:#334155;">
The main objective is to simulate a realistic <b>mid-course prediction environment</b>, where the model can analyze student behavior before the course is completed. Unlike full-course prediction approaches, this notebook intentionally restricts the available information to the first half of the learning journey to support earlier educational intervention and risk detection.
</p>

<hr style="border:0; border-top:1px solid #bfdbfe; margin:24px 0;">

<h2 style="color:#0f172a; font-size:26px;">
🎯 Dataset Construction
</h2>

<div style="background:#ffffff; border:1px solid #bfdbfe; border-radius:14px; padding:18px 22px; margin:14px 0 22px 0; box-shadow:0 2px 8px rgba(37,99,235,0.08);">

<p style="font-size:16px; margin-top:0; color:#1e40af;"><b>The generated dataset combines:</b></p>

<ul style="font-size:16px; line-height:2; margin-bottom:0;">
  <li>Demographic background</li>
  <li>Academic performance</li>
  <li>VLE engagement behavior</li>
  <li>Assessment activity</li>
  <li>Workload indicators</li>
  <li>Arab-semester normalized timing features</li>
</ul>

</div>

<p style="font-size:17px; line-height:1.8; color:#334155;">
At this stage, the dataset captures a stronger representation of student learning patterns compared with the first-quarter dataset, allowing the system to identify deeper behavioral and academic signals related to student risk, disengagement, or potential dropout.
</p>

<hr style="border:0; border-top:1px solid #bfdbfe; margin:24px 0;">

<h2 style="color:#0f172a; font-size:26px;">
📦 Final Output
</h2>

<div style="background:#ffffff; border:1px solid #bfdbfe; border-radius:14px; padding:18px 22px; margin:14px 0 22px 0; box-shadow:0 2px 8px rgba(37,99,235,0.08);">

<p style="font-size:16px; line-height:1.8; color:#334155; margin:0;">
The final output of this notebook is 
<code style="background:#e0f2fe; color:#075985; padding:3px 7px; border-radius:6px;">Selected_features_Q2.csv</code>, 
which contains the selected mid-course features used in the next modeling and evaluation stages of the AI-based early-warning system.
</p>

</div>

</div>

<h2 style="color:#EA580C;">📥 Load Processed Input Tables</h2>

This section loads the processed tables required to build the 50% early-warning dataset.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path


ARAB_SEMESTER_LENGTH = 150

base = pd.read_csv("C:\\Users\\HP\\OneDrive\\المستندات\\GitHub\\Student-Leak-Radar\\project\\archive\\data\\processed\\base_student_table.csv")
student_vle_full = pd.read_csv("C:\\Users\\HP\\OneDrive\\المستندات\\GitHub\\Student-Leak-Radar\\project\\archive\\data\\processed\\student_vle_full.csv")
assessment_full = pd.read_csv("C:\\Users\\HP\\OneDrive\\المستندات\\GitHub\\Student-Leak-Radar\\project\\archive\\data\\processed\\assessment_full.csv")

<h2 style="color:#EA580C;">🧩 Build Early-Warning Dataset Function</h2>

This function creates an early-warning dataset using only student activity and assessment data available up to a selected course-progress cutoff.

In [ ]:
def build_early_warning_dataset(cutoff, output_name):
    """
    Build an early-warning dataset using only events available up to a given course-progress cutoff.
    """

    # Filter VLE interactions up to the selected cutoff.
    vle_until_cutoff = student_vle_full[
        student_vle_full["relative_time"] <= cutoff
    ].copy()

    # Aggregate VLE engagement features up to the cutoff.
    vle_features = vle_until_cutoff.groupby(
        ["id_student", "code_module", "code_presentation"]
    ).agg(
        total_clicks_until_cutoff=("sum_click", "sum"),
        active_days_until_cutoff=("date", "nunique"),
        vle_module_presentation_length=("module_presentation_length", "first"),
        unique_sites_until_cutoff=("id_site", "nunique"),
        unique_activity_types_until_cutoff=("activity_type", "nunique")
    ).reset_index()

    # Calculate click intensity per active day.
    vle_features["clicks_per_active_day_until_cutoff"] = (
        vle_features["total_clicks_until_cutoff"] /
        vle_features["active_days_until_cutoff"]
    ).replace([np.inf, -np.inf], 0).fillna(0)

    # Calculate active-day ratio relative to course length.
    vle_features["active_days_ratio_until_cutoff"] = (
        vle_features["active_days_until_cutoff"] /
        vle_features["vle_module_presentation_length"]
    ).replace([np.inf, -np.inf], 0).fillna(0)

    # Map active-day ratio to the 150-day Arab semester scale.
    vle_features["arab_active_days_equivalent_until_cutoff"] = (
        vle_features["active_days_ratio_until_cutoff"] * ARAB_SEMESTER_LENGTH
    )

    # Build activity-type click features up to the cutoff.
    activity_features = vle_until_cutoff.pivot_table(
        index=["id_student", "code_module", "code_presentation"],
        columns="activity_type",
        values="sum_click",
        aggfunc="sum",
        fill_value=0
    ).reset_index()

    activity_features.columns.name = None

    # Filter assessment submissions up to the selected cutoff.
    assessment_until_cutoff = assessment_full[
        assessment_full["submitted_relative_time"] <= cutoff
    ].copy()

    # Aggregate assessment performance and submission features up to the cutoff.
    assessment_features = assessment_until_cutoff.groupby(
        ["id_student", "code_module", "code_presentation"]
    ).agg(
        avg_score_until_cutoff=("score", "mean"),
        min_score_until_cutoff=("score", "min"),
        max_score_until_cutoff=("score", "max"),
        submitted_assessments_until_cutoff=("id_assessment", "count"),
        late_submissions_until_cutoff=("is_late", "sum"),
        avg_submission_delay_until_cutoff=("submission_delay", "mean"),
        avg_assessment_relative_time_until_cutoff=("assessment_relative_time", "mean"),
        avg_submitted_relative_time_until_cutoff=("submitted_relative_time", "mean"),
        avg_assessment_arab_day_until_cutoff=("assessment_arab_day", "mean"),
        avg_submitted_arab_day_until_cutoff=("submitted_arab_day", "mean"),
        avg_submission_delay_relative_until_cutoff=("submission_delay_relative", "mean"),
        avg_submission_delay_arab_days_until_cutoff=("submission_delay_arab_days", "mean")
    ).reset_index()

    # Merge VLE, activity-type, and assessment features with the base table.
    final_early_df = base.merge(
        vle_features,
        on=["id_student", "code_module", "code_presentation"],
        how="left"
    )

    final_early_df = final_early_df.merge(
        activity_features,
        on=["id_student", "code_module", "code_presentation"],
        how="left"
    )

    final_early_df = final_early_df.merge(
        assessment_features,
        on=["id_student", "code_module", "code_presentation"],
        how="left"
    )

    # Fill missing numerical values with zero.
    numeric_cols = final_early_df.select_dtypes(include=["number"]).columns
    final_early_df[numeric_cols] = final_early_df[numeric_cols].fillna(0)

    # Fill missing deprivation-band values.
    final_early_df["imd_band"] = final_early_df["imd_band"].fillna("Unknown")

    # Create workload features.
    FULL_LOAD_OU = 120

    final_early_df["workload_ratio"] = (
        final_early_df["studied_credits"] / FULL_LOAD_OU
    ).clip(upper=2.0)

    final_early_df["workload_level"] = pd.cut(
        final_early_df["workload_ratio"],
        bins=[0, 0.5, 1.0, 1.5, 2.0],
        labels=["light", "normal", "high", "very_high"],
        include_lowest=True
    )

    # Save the generated early-warning dataset.
    from pathlib import Path

    output_path = Path(f"data/processed/{output_name}")
    output_path.parent.mkdir(parents=True, exist_ok=True)

    final_early_df.to_csv(output_path, index=False)

    print(f"Saved: {output_path}")
    print("Shape:", final_early_df.shape)

    return final_early_df

<div style="background:#fff7ed; border:1px solid #fdba74; border-left:5px solid #ea580c; padding:16px 20px; border-radius:12px; font-family:Segoe UI, Arial, sans-serif; color:#1e293b;">

<p style="font-size:16px; margin-top:0; color:#c2410c;">
<b>🔎 Observation</b>
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155;">
The function is now ready to generate early-warning datasets based on any selected course-progress cutoff.
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155; margin-bottom:0;">
It keeps only VLE interactions and assessment submissions available before the cutoff, then rebuilds the student-level feature table for early prediction.
</p>

</div>

<h2 style="color:#EA580C;">⏱️ Generate the 50% Early-Warning Dataset</h2>

This section builds an early-warning dataset using only the first 50% of the course timeline.

In [21]:
# Build the 50% early-warning dataset using only data available
# during the first half of the course timeline.
student_features_first_50 = build_early_warning_dataset(
    cutoff=0.50,
    output_name="student_features_first_50.csv"
)

Saved: data\processed\student_features_first_50.csv
Shape: (32480, 55)


<div style="background:#fff7ed; border:1px solid #fdba74; border-left:5px solid #ea580c; padding:16px 20px; border-radius:12px; font-family:Segoe UI, Arial, sans-serif; color:#1e293b;">

<p style="font-size:16px; margin-top:0; color:#c2410c;">
<b>🔎 Observation</b>
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155;">
The 50% early-warning dataset was generated successfully.
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155; margin-bottom:0;">
This dataset represents student behavior, assessment activity, and engagement patterns available during the first half of the course.
</p>

</div>

<h2 style="color:#EA580C;">🔍 Reload and Review the 50% Dataset</h2>

This section reloads the saved 50% early-warning dataset to confirm that the file was exported correctly.

In [22]:
# Reload the saved 50% early-warning dataset.
student_features_first_50 = pd.read_csv(
    r"C:\Users\HP\OneDrive\المستندات\GitHub\Student-Leak-Radar\project\data\processed\student_features_first_50.csv"
)

# head the dataset.
student_features_first_50.head()

,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,disability,...,late_submissions_until_cutoff,avg_submission_delay_until_cutoff,avg_assessment_relative_time_until_cutoff,avg_submitted_relative_time_until_cutoff,avg_assessment_arab_day_until_cutoff,avg_submitted_arab_day_until_cutoff,avg_submission_delay_relative_until_cutoff,avg_submission_delay_arab_days_until_cutoff,workload_ratio,workload_level
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,N,...,0.0,-1.333333,0.236318,0.231343,35.447761,34.701493,-0.004975,-0.746269,2.0,very_high
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,N,...,2.0,1.666667,0.236318,0.242537,35.447761,36.380597,0.006219,0.932836,0.5,light
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,Y,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.5,light
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,N,...,0.0,-2.333333,0.236318,0.227612,35.447761,34.141791,-0.008706,-1.305970,0.5,light
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,N,...,3.0,11.666667,0.236318,0.279851,35.447761,41.977612,0.043532,6.529851,0.5,light


<h2 style="color:#EA580C;">🧾 Review 50% Dataset Columns</h2>

This section reviews the columns available in the generated 50% early-warning dataset.

In [23]:
# Display all columns in the 50% early-warning dataset.
student_features_first_50.columns

Index(['code_module', 'code_presentation', 'id_student', 'gender', 'region',
       'highest_education', 'imd_band', 'age_band', 'num_of_prev_attempts',
       'disability', 'final_result', 'target', 'module_presentation_length',
       'active_days_until_cutoff', 'vle_module_presentation_length',
       'unique_sites_until_cutoff', 'unique_activity_types_until_cutoff',
       'clicks_per_active_day_until_cutoff', 'active_days_ratio_until_cutoff',
       'arab_active_days_equivalent_until_cutoff', 'dataplus', 'dualpane',
       'externalquiz', 'forumng', 'glossary', 'homepage', 'htmlactivity',
       'oucollaborate', 'oucontent', 'ouelluminate', 'ouwiki', 'page',
       'questionnaire', 'quiz', 'repeatactivity', 'resource', 'sharedsubpage',
       'subpage', 'url', 'avg_score_until_cutoff',
       'submitted_assessments_until_cutoff', 'late_submissions_until_cutoff',
       'avg_submission_delay_until_cutoff',
       'avg_assessment_relative_time_until_cutoff',
       'avg_submitted_re

<div style="background:#fff7ed; border:1px solid #fdba74; border-left:5px solid #ea580c; padding:16px 20px; border-radius:12px; font-family:Segoe UI, Arial, sans-serif; color:#1e293b;">

<p style="font-size:16px; margin-top:0; color:#c2410c;">
<b>🔎 Observation</b>
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155;">
The 50% early-warning dataset contains a rich set of demographic, academic, engagement, behavioral, timing, and workload-related features.
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155;">
The dataset includes normalized activity measures, assessment timing indicators, Arab-semester time features, and VLE activity-type interactions, which provide a stronger mid-course representation of student learning behavior.
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155; margin-bottom:0;">
These features will support the next stages of correlation analysis, feature selection, and early-risk prediction modeling.
</p>

</div>

<h2 style="color:#EA580C;">📊 Inspect the 50% Early-Warning Dataset</h2>

This section reviews the structure and dimensions of the 50% early-warning dataset before starting feature analysis and selection.

In [14]:
student_features_first_50 = student_features_first_50.copy()

print(student_features_first_50.shape)

student_features_first_50.head()

(32480, 51)


,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,disability,...,late_submissions_until_cutoff,avg_submission_delay_until_cutoff,avg_assessment_relative_time_until_cutoff,avg_submitted_relative_time_until_cutoff,avg_assessment_arab_day_until_cutoff,avg_submitted_arab_day_until_cutoff,avg_submission_delay_relative_until_cutoff,avg_submission_delay_arab_days_until_cutoff,workload_ratio,workload_level
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,N,...,0.0,-1.333333,0.236318,0.231343,35.447761,34.701493,-0.004975,-0.746269,2.0,very_high
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,N,...,2.0,1.666667,0.236318,0.242537,35.447761,36.380597,0.006219,0.932836,0.5,light
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,Y,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.5,light
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,N,...,0.0,-2.333333,0.236318,0.227612,35.447761,34.141791,-0.008706,-1.305970,0.5,light
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,N,...,3.0,11.666667,0.236318,0.279851,35.447761,41.977612,0.043532,6.529851,0.5,light


<div style="background:#fff7ed; border:1px solid #fdba74; border-left:5px solid #ea580c; padding:16px 20px; border-radius:12px; font-family:Segoe UI, Arial, sans-serif; color:#1e293b;">

<p style="font-size:16px; margin-top:0; color:#c2410c;">
<b>🔎 Observation</b>
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155;">
The 50% early-warning dataset contains 
<code style="background:#ffedd5; color:#9a3412; padding:3px 7px; border-radius:6px;">32,480</code> 
student-course records and 
<code style="background:#ffedd5; color:#9a3412; padding:3px 7px; border-radius:6px;">51</code> 
features.
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155;">
Each row represents a single student within a specific course presentation, while the columns describe demographic background, learning behavior, assessment activity, VLE engagement patterns, normalized timing features, and workload indicators.
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155; margin-bottom:0;">
The dataset structure confirms that the feature-engineering and aggregation pipeline was completed successfully before feature-selection and modeling stages.
</p>

</div>

<h2 style="color:#EA580C;">🔢 Select Numeric Features for Correlation Analysis</h2>

This section selects only numerical columns from the 50% early-warning dataset and removes identifier columns that should not be used in correlation analysis.

In [ ]:
# Select only numerical columns from the 50% early-warning dataset.
numeric_df_50 = student_features_first_50.select_dtypes(include=["number"]).copy()

# Remove id_student because it is an identifier, not a meaningful predictive feature.
if "id_student" in numeric_df_50.columns:
    numeric_df_50 = numeric_df_50.drop(columns=["id_student"])

<div style="background:#fff7ed; border:1px solid #fdba74; border-left:5px solid #ea580c; padding:16px 20px; border-radius:12px; font-family:Segoe UI, Arial, sans-serif; color:#1e293b;">

<p style="font-size:16px; margin-top:0; color:#c2410c;">
<b>🔎 Observation</b>
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155;">
The numeric feature subset was prepared for correlation analysis.
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155; margin-bottom:0;">
The 
<code style="background:#ffedd5; color:#9a3412; padding:3px 7px; border-radius:6px;">id_student</code> 
column was removed because it is only a student identifier and does not represent a behavioral, academic, or engagement feature.
</p>

</div>

<h2 style="color:#EA580C;">📈 Compute Feature Correlation Matrix</h2>

This section calculates the absolute Pearson correlation matrix for all numerical features in the 50% early-warning dataset.

In [24]:
# Compute the absolute Pearson correlation matrix
# for all numerical features.
corr_matrix_50 = numeric_df_50.corr().abs()

<div style="background:#fff7ed; border:1px solid #fdba74; border-left:5px solid #ea580c; padding:16px 20px; border-radius:12px; font-family:Segoe UI, Arial, sans-serif; color:#1e293b;">

<p style="font-size:16px; margin-top:0; color:#c2410c;">
<b>🔎 Observation</b>
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155;">
The correlation matrix measures the strength of linear relationships between numerical features.
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155; margin-bottom:0;">
Absolute correlation values were used so that both strong positive and strong negative relationships can be analyzed equally during feature-selection and multicollinearity analysis.
</p>

</div>

<h2 style="color:#EA580C;">🔥 Visualize Correlation Heatmap</h2>

This section visualizes the correlation matrix to identify highly related numerical features in the 50% early-warning dataset.

In [25]:
import plotly.graph_objects as go

# Create an interactive heatmap for the 50% feature correlation matrix.
fig = go.Figure(
    data=go.Heatmap(
        z=corr_matrix_50.values,
        x=corr_matrix_50.columns,
        y=corr_matrix_50.columns,
        colorscale="RdBu",
        zmin=-1,
        zmax=1
    )
)

# Customize the heatmap layout.
fig.update_layout(
    title="Correlation Heatmap - First 50%",
    width=1000,
    height=900
)

fig.show()

<div style="background:#fff7ed; border:1px solid #fdba74; border-left:5px solid #ea580c; padding:16px 20px; border-radius:12px; font-family:Segoe UI, Arial, sans-serif; color:#1e293b;">

<p style="font-size:16px; margin-top:0; color:#c2410c;">
<b>🔎 Observation</b>
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155;">
The heatmap provides a visual overview of relationships between numerical features in the 50% early-warning dataset.
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155; margin-bottom:0;">
Strong correlations appear among several engagement, timing, and assessment-related features, which helps identify potentially redundant columns before final feature selection.
</p>

</div>

<h2 style="color:#EA580C;">🧹 Remove Manually Selected Redundant Features</h2>

This section removes selected highly correlated or less useful numerical features from the 50% early-warning dataset.

In [26]:
# Define features to remove manually based on correlation analysis
# and feature relevance for early-warning modeling.
manual_drop_50 = [
    "max_score_until_cutoff",
    "min_score_until_cutoff",
    "studied_credits",
    "total_clicks_until_cutoff"
]

# Drop selected redundant features if they exist.
student_features_first_50 = student_features_first_50.drop(
    columns=manual_drop_50,
    errors="ignore"
)

# Display the updated dataset shape.
print(student_features_first_50.shape)

(32480, 51)


<div style="background:#fff7ed; border:1px solid #fdba74; border-left:5px solid #ea580c; padding:16px 20px; border-radius:12px; font-family:Segoe UI, Arial, sans-serif; color:#1e293b;">

<p style="font-size:16px; margin-top:0; color:#c2410c;">
<b>🔎 Observation</b>
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155;">
The selected redundant features were removed from the 50% early-warning dataset when available.
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155; margin-bottom:0;">
The dataset shape remained 
<code style="background:#ffedd5; color:#9a3412; padding:3px 7px; border-radius:6px;">(32480, 51)</code>, 
which means these columns were either already removed or not present in the current dataset.
</p>

</div>

<h2 style="color:#EA580C;">💾 Save the Final 50% Early-Warning Dataset</h2>

This section saves the cleaned and feature-selected 50% early-warning dataset for the next modeling stages.

In [27]:
from pathlib import Path

# Create the processed-data directory if it does not exist.
Path("data/processed").mkdir(
    parents=True,
    exist_ok=True
)

# Save the final 50% early-warning dataset.
student_features_first_50.to_csv(
    "data/processed/student_features_first_50.csv",
    index=False
)

print("student_features_first_50 saved successfully.")

student_features_first_50 saved successfully.


<div style="background:#fff7ed; border:1px solid #fdba74; border-left:5px solid #ea580c; padding:16px 20px; border-radius:12px; font-family:Segoe UI, Arial, sans-serif; color:#1e293b;">

<p style="font-size:16px; margin-top:0; color:#c2410c;">
<b>🔎 Observation</b>
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155;">
The final 50% early-warning dataset was saved successfully in the processed-data folder.
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155; margin-bottom:0;">
The dataset now contains cleaned, aggregated, and feature-selected information that is ready for machine-learning experiments and mid-course early-risk prediction tasks.
</p>

</div>

<h2 style="color:#EA580C;">🎯 Select Final Features for Q2 Model</h2>

This section selects the final feature subset for the Q2 early-warning model based on correlation analysis, feature relevance, and domain knowledge.

In [28]:
# Select the final features that will be used in the Q2 early-warning model.
# These features were chosen based on correlation analysis and domain knowledge.
selected_features_q2 = [
    "avg_score_until_cutoff",
    "submitted_assessments_until_cutoff",
    "arab_active_days_equivalent_until_cutoff",
    "avg_submission_delay_arab_days_until_cutoff",
    "homepage",
    "forumng",
    "unique_sites_until_cutoff",
    "unique_activity_types_until_cutoff",
    "clicks_per_active_day_until_cutoff",
    "resource",
    "subpage",
    "url",
    "oucontent",
    "quiz",
    "highest_education"
]

# Check that all selected features exist in the dataset.
missing_features = [
    column for column in selected_features_q2
    if column not in student_features_first_50.columns
]

if missing_features:
    raise ValueError(f"Missing selected features: {missing_features}")

# Create the final selected-feature dataset with the target column.
selected_df_q2 = student_features_first_50[selected_features_q2 + ["target"]]

# Save the selected Q2 features dataset.
selected_df_q2.to_csv(
    "data/processed/Selected_features_Q2.csv",
    index=False
)

print("Selected_features_Q2 saved successfully.")
print("Shape:", selected_df_q2.shape)

Selected_features_Q2 saved successfully.
Shape: (32480, 16)


<div style="background:#fff7ed; border:1px solid #fdba74; border-left:5px solid #ea580c; padding:16px 20px; border-radius:12px; font-family:Segoe UI, Arial, sans-serif; color:#1e293b;">

<p style="font-size:16px; margin-top:0; color:#c2410c;">
<b>🔎 Observation</b>
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155;">
The final Q2 feature subset was saved successfully as 
<code style="background:#ffedd5; color:#9a3412; padding:3px 7px; border-radius:6px;">Selected_features_Q2.csv</code>.
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155;">
The selected features combine academic performance, mid-course engagement, Arab-normalized time features, VLE activity behavior, and demographic background.
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155; margin-bottom:0;">
This reduced dataset is ready to be used for the second-quarter early-warning modeling stage.
</p>

</div>

<div style="background:#f8fafc; border:1px solid #dbeafe; border-left:7px solid #2563eb; padding:26px 30px; border-radius:16px; font-family:Segoe UI, Arial, sans-serif; color:#1e293b;">

<h2 style="margin-top:0; color:#1d4ed8; font-size:30px;">
✅ Notebook Summary
</h2>

<p style="font-size:17px; line-height:1.8; color:#334155;">
This notebook successfully generated the <b>second-quarter early-warning dataset</b> for student risk prediction using only the first <b>50% of the course timeline</b>.
</p>

<p style="font-size:17px; line-height:1.8; color:#334155;">
The workflow included loading processed intermediate tables, generating the 50% cutoff dataset, reviewing dataset structure and feature availability, preparing numerical features, analyzing correlations, identifying redundant columns, and selecting the final Q2 feature subset.
</p>

<hr style="border:0; border-top:1px solid #bfdbfe; margin:24px 0;">

<h2 style="color:#0f172a; font-size:26px;">
📌 Final Dataset Value
</h2>

<div style="background:#ffffff; border:1px solid #bfdbfe; border-radius:14px; padding:18px 22px; margin:14px 0 22px 0; box-shadow:0 2px 8px rgba(37,99,235,0.08);">

<p style="font-size:16px; line-height:1.8; color:#334155; margin-top:0;">
The resulting dataset combines <b>demographic information</b>, academic performance indicators, VLE engagement behavior, timing-related features, and Arab-semester normalized activity measures to provide a realistic mid-course representation of student learning behavior.
</p>

<p style="font-size:16px; line-height:1.8; color:#334155; margin-bottom:0;">
Compared with the first-quarter dataset, the Q2 dataset captures richer behavioral and assessment patterns, making it more suitable for stronger early-risk prediction and educational intervention scenarios.
</p>

</div>

<p style="font-size:17px; line-height:1.8; color:#334155;">
The final output, <code style="background:#e0f2fe; color:#075985; padding:3px 7px; border-radius:6px;">Selected_features_Q2.csv</code>, is now fully prepared for the next stage of machine-learning experimentation, model training, and evaluation within the AI-based early-warning system.
</p>

</div>